# Random Forest Training

21 experiments: 7 feature sets × 3 split strategies.
1. 5-fold Optuna (TPE) hyperparameter search on the train set vs. 5-fold Random
2. Refit best params on full train set
3. Evaluate on test set (RMSE, MAE, R²)

In [ ]:
import sys, pathlib, os, json, time, warnings
import numpy as np
import pandas as pd
import joblib
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import randint

warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# project root
ROOT = pathlib.Path(".").resolve().parent
sys.path.insert(0, str(ROOT))
from src.data_utils import load_experiment

MODELS_DIR  = ROOT / "models" / "rf"
RESULTS_DIR = ROOT / "results" / "rf"

TARGET_IDX = 2  # co2_mol_kg_0.1bar
SEED = 42

In [ ]:
# configuration
FEATURE_SETS = [
    "baseline",
    "geo_decorr", "geo_all",
    "rac_decorr", "rac_all",
    "combined_decorr", "combined_all",
]

SPLITS = ["random", "topology", "metal"]

N_TRIALS = 30  # Optuna trials
N_ITER   = 30  # RandomizedSearch iterations
CV_FOLDS = 5

# RandomizedSearch search space
PARAM_DIST = {
    "n_estimators":      randint(100, 800),
    "max_depth":         [None, 10, 20, 30, 40],
    "min_samples_split": randint(2, 20),
    "min_samples_leaf":  randint(1, 20),
    "max_features":      ["sqrt", "log2", 0.3, 0.5, 0.7, 1.0],
}

print(f"Experiments: {len(FEATURE_SETS)} feature sets × {len(SPLITS)} splits = {len(FEATURE_SETS)*len(SPLITS)}")
print(f"Per experiment: {CV_FOLDS} folds × {N_TRIALS} configs = {CV_FOLDS*N_TRIALS} fits")

Training loop

In [ ]:
def _search_optuna(X, y):
    kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

    def objective(trial):
        use_max_depth = trial.suggest_categorical("use_max_depth", [True, False])
        params = {
            "n_estimators":      trial.suggest_int("n_estimators", 100, 800),
            "max_depth":         trial.suggest_int("max_depth", 5, 40) if use_max_depth else None,
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 20),
            "max_features":      trial.suggest_float("max_features", 0.1, 1.0),
        }
        model = RandomForestRegressor(random_state=SEED, n_jobs=-1, **params)
        scores = cross_val_score(model, X, y, cv=kf, scoring="neg_mean_squared_error")
        return float(np.sqrt(-scores.mean()))

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

    bt = study.best_trial
    best_params = {
        "n_estimators":      bt.params["n_estimators"],
        "max_depth":         bt.params.get("max_depth") if bt.params["use_max_depth"] else None,
        "min_samples_split": bt.params["min_samples_split"],
        "min_samples_leaf":  bt.params["min_samples_leaf"],
        "max_features":      bt.params["max_features"],
    }
    return best_params, study.best_value


def _search_random(X, y):
    rf = RandomForestRegressor(random_state=SEED, n_jobs=-1)
    search = RandomizedSearchCV(
        rf,
        param_distributions=PARAM_DIST,
        n_iter=N_ITER,
        cv=CV_FOLDS,
        scoring="neg_mean_squared_error",
        random_state=SEED,
        n_jobs=-1,
        refit=False,
        verbose=0,
    )
    search.fit(X, y)
    return search.best_params_, float(np.sqrt(-search.best_score_))


def _to_json_val(v):
    if v is None:            return None
    if isinstance(v, bool):  return bool(v)
    if isinstance(v, str):   return v
    if isinstance(v, float): return float(v)
    return int(v)


def run_experiment(feature_set, split, method="optuna"):
    tag = f"{feature_set}__{split}"
    model_dir   = MODELS_DIR / method
    results_dir = RESULTS_DIR / method
    model_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)

    model_path  = model_dir  / f"{tag}.joblib"
    params_path = results_dir / f"{tag}_params.json"

    X_train, X_test, y_train, y_test = load_experiment(
        split_strategy=split, feature_set=feature_set
    )
    y_tr = y_train[:, TARGET_IDX]
    y_te = y_test[:,  TARGET_IDX]

    # Only use checkpoint if BOTH files are present
    if model_path.exists() and params_path.exists():
        print(f"  LOAD {tag} [{method}] (model exists, re-evaluating)", end="  ", flush=True)
        best = joblib.load(model_path)
        with open(params_path) as f:
            best_params = json.load(f)
        kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
        cv_model = RandomForestRegressor(random_state=SEED, n_jobs=-1, **best_params)
        cv_scores = cross_val_score(cv_model, X_train, y_tr, cv=kf, scoring="neg_mean_squared_error")
        best_cv_rmse = float(np.sqrt(-cv_scores.mean()))
    else:
        print(f"  {tag} [{method}]: X_train {X_train.shape} ...", end="  ", flush=True)

        if method == "optuna":
            best_params, best_cv_rmse = _search_optuna(X_train, y_tr)
        else:
            best_params, best_cv_rmse = _search_random(X_train, y_tr)

        best = RandomForestRegressor(random_state=SEED, n_jobs=-1, **best_params)
        best.fit(X_train, y_tr)

        joblib.dump(best, model_path)
        with open(params_path, "w") as f:
            json.dump({k: _to_json_val(v) for k, v in best_params.items()}, f, indent=2)

    y_pred    = best.predict(X_test)
    test_rmse = float(np.sqrt(mean_squared_error(y_te, y_pred)))
    test_mae  = float(mean_absolute_error(y_te, y_pred))
    test_r2   = float(r2_score(y_te, y_pred))

    result = {
        "feature_set":  feature_set,
        "split":        split,
        "method":       method,
        "n_features":   X_train.shape[1],
        "best_cv_RMSE": best_cv_rmse,
        "test_RMSE":    test_rmse,
        "test_MAE":     test_mae,
        "test_R2":      test_r2,
    }
    print(f"CV RMSE={best_cv_rmse:.4f}  |  test RMSE={test_rmse:.4f}  MAE={test_mae:.4f}  R²={test_r2:.4f}")
    return result

In [ ]:
%%time

results = []
SUMMARY_PATH = RESULTS_DIR / "rf_summary.csv"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for method in ["optuna", "random"]:
    print(f"\n{'='*50}")
    print(f"  Method: {method}")
    print(f"{'='*50}")
    for i, (fs, sp) in enumerate(
        [(fs, sp) for fs in FEATURE_SETS for sp in SPLITS], 1
    ):
        print(f"\n[{i}/21]", end=" ")
        t0 = time.time()
        r = run_experiment(fs, sp, method=method)
        if r is not None:
            r["time_min"] = round((time.time() - t0) / 60, 1)
            results.append(r)
            pd.DataFrame(results).to_csv(SUMMARY_PATH, index=False)

print(f"\nDone. {len(results)} experiments saved.")

Results summary

In [ ]:
df_res = pd.read_csv(RESULTS_DIR / "rf_summary.csv")

# CV RMSE
print("CV RMSE (5-fold on train set)")
pivot_cv = df_res.pivot_table(index="feature_set", columns=["split", "method"], values="best_cv_RMSE")
pivot_cv = pivot_cv.reindex(FEATURE_SETS)
display(pivot_cv.style.format("{:.4f}").background_gradient(cmap="RdYlGn_r"))

# Test RMSE
print("\nTest RMSE")
pivot_test_rmse = df_res.pivot_table(index="feature_set", columns=["split", "method"], values="test_RMSE")
pivot_test_rmse = pivot_test_rmse.reindex(FEATURE_SETS)
display(pivot_test_rmse.style.format("{:.4f}").background_gradient(cmap="RdYlGn_r"))

# Test R²
print("\nTest R²")
pivot_r2 = df_res.pivot_table(index="feature_set", columns=["split", "method"], values="test_R2")
pivot_r2 = pivot_r2.reindex(FEATURE_SETS)
display(pivot_r2.style.format("{:.4f}").background_gradient(cmap="RdYlGn"))

In [ ]:
# Optuna-only view (matches XGBoost results notebook layout)
df_opt = df_res[df_res["method"] == "optuna"]

print("CV RMSE (5-fold on train set) — Optuna")
pivot_cv = df_opt.pivot_table(index="feature_set", columns="split", values="best_cv_RMSE")
pivot_cv = pivot_cv.reindex(FEATURE_SETS)
display(pivot_cv.style.format("{:.4f}").background_gradient(cmap="RdYlGn_r"))

print("\nTest RMSE — Optuna")
pivot_test_rmse = df_opt.pivot_table(index="feature_set", columns="split", values="test_RMSE")
pivot_test_rmse = pivot_test_rmse.reindex(FEATURE_SETS)
display(pivot_test_rmse.style.format("{:.4f}").background_gradient(cmap="RdYlGn_r"))

print("\nTest R² — Optuna")
pivot_r2 = df_opt.pivot_table(index="feature_set", columns="split", values="test_R2")
pivot_r2 = pivot_r2.reindex(FEATURE_SETS)
display(pivot_r2.style.format("{:.4f}").background_gradient(cmap="RdYlGn"))